# Criação da tabela Silver - STATISTICS

## Objetivos
- Criação de tabela que capture dados da bronze sem ocorreência de duplicatas e apenas de partidas encerradas
- Explode do array "statistics" que contém as informações das partidas
- Uso das informações do array para a criação de novas colunas, filtrando apenas tipos desejados
- Criação de registros separados para o time mandante e visitante
- União de dataframes e uso de pivot
- Criação de colunas calculadas

> Nas estatísticas haviam casos de registros duplicados. Para esses casos, foi mantido o maior valor.

In [0]:
from pyspark.sql.functions import *

In [0]:
max_dh_bronze = (
    spark.read.table("api_football.bronze.matches_raw")
    # variável que captura a data mais recente de atualização (maior)
    .agg(max("dh_processing_br").alias("max_dh_bronze"))
    # transforma de dataframe para variável
    .collect()[0]["max_dh_bronze"]
)

# leitura da tabela de comparação, nesse caso a statistics
df_silver = spark.read.table("api_football.silver.statistics")

# join para pegar os dados que não estão na silver
df_matches_raw = (
    spark.read.table("api_football.bronze.matches_raw")
    .alias("mr")
    # left anti pega os dados das match_id que estão na bronze, mas não estão na silver
    .join(df_silver.alias("s"), col("s.match_id") == col("mr.match_id"), "leftanti")
    # filtra somente os dados que tem a data de atualização mais recente
    .filter(col("mr.dh_processing_br") == max_dh_bronze)
    # filtra somente as partidas finalizadas
    .filter(col("mr.match_status") == "Finished")
)

In [0]:
df_statistics_exploded = df_matches_raw.select(
    "match_id",
    "match_hometeam_id",
    "match_hometeam_name",
    "match_awayteam_id",
    "match_awayteam_name",
    explode("statistics").alias("stat")
)

# display(df_statistics_exploded)

In [0]:
# variável que estabelece campos do array que serão utilizados
statistics_required = [
    "Corners",
    "Penalty",
    "Substitution",
    "Attacks",
    "Shots Total",
    "Shots On Goal",
    "Shots Off Goal",
    "Shots Blocked",
    "Fouls",
    "Offsides",
    "Passes Total",
    "Passes Accurate"
]

# seleciona apenas os dados da coluna stat em que o campo type está na lista statistics_required
df_statistics_filtered = df_statistics_exploded.filter(
    col("stat.type").isin(statistics_required)
)

# display(df_statistics_filtered)

In [0]:
df_home = df_statistics_filtered.select(
    col("match_id"),
    col("match_hometeam_id").alias("team_id"),
    col("match_hometeam_name").alias("team_name"),
    # separa o campo type em uma nova coluna
    col("stat.type").alias("statistic_type"),
    # seleciona apenas campos marcados como "home" da tabela stats e transforma em inteiro
    col("stat.home").try_cast("int").alias("statistic_value")
)

# display(df_home)

In [0]:
df_away = df_statistics_filtered.select(
    col("match_id"),
    col("match_awayteam_id").alias("team_id"),
    col("match_awayteam_name").alias("team_name"),
    # separa o campo type em uma nova coluna
    col("stat.type").alias("statistic_type"),
    # seleciona apenas campos marcados como "away" da tabela stats e transforma em inteiro
    col("stat.away").try_cast("int").alias("statistic_value")
)

# display (df_away)

In [0]:
# união dos dois dataframes
df_statistics_long = (
    df_home.unionByName(df_away)
    # ordenação para verificação
).orderBy("match_id", "team_id", "statistic_type")


# display(df_statistics_long)

In [0]:
df_statistics = (
    # agrupamento devido a agregação
    df_statistics_long.groupBy("match_id", "team_id", "team_name")
    # pivot que cria uma coluna para cada tipo de estatística baseado no que está armazenado na variável statistics_required a partir da coluna statistic_type
    .pivot("statistic_type", statistics_required)
    # seleciona o valor máximo de uma estatística para remover casos de duplicidade
    .agg(max("statistic_value"))
).orderBy("match_id", "team_id")

In [0]:
df_statistics = (
    df_statistics.withColumnRenamed("Corners", "corners")
    .withColumnRenamed("Penalty", "penalty")
    .withColumnRenamed("Substitution", "substitutions")
    .withColumnRenamed("Attacks", "attacks")
    .withColumnRenamed("Ball Possession", "ball_possession")
    .withColumnRenamed("Shots Total", "shots_total")
    .withColumnRenamed("Shots On Goal", "shots_on_goal")
    .withColumnRenamed("Shots Off Goal", "shots_off_goal")
    .withColumnRenamed("Shots Blocked", "shots_blocked")
    .withColumnRenamed("Fouls", "fouls")
    .withColumnRenamed("Offsides", "offsides")
    .withColumnRenamed("Passes Total", "passes_total")
    .withColumnRenamed("Passes Accurate", "passes_accurate")
)

# display(df_statistics)

In [0]:
df_statistics.write.mode("append").saveAsTable("api_football.silver.statistics")

In [0]:
%sql
select
  *
from
  api_football.silver.statistics